In [1]:
import socket

def send_cp1251_message_with_response(message_text, host='localhost', port=5999, timeout=5, buffer_size=1024):
    """
    Отправляет строку в cp1251 и возвращает ответ от сервера

    Параметры:
    ----------
    message_text : str
        Строка для отправки
    host : str, optional
        Адрес сервера
    port : int, optional
        Порт сервера
    timeout : int, optional
        Таймаут ожидания ответа в секундах
    buffer_size : int, optional
        Размер буфера для приема данных

    Возвращает:
    -----------
    tuple (bool, str)
        (успех отправки, ответ от сервера или сообщение об ошибке)
    """
    client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client_socket.settimeout(timeout)

    try:
        # Подключаемся к серверу
        client_socket.connect((host, port))

        # Отправляем сообщение
        client_socket.send(message_text.encode('cp1251'))
        print(f"Отправлено: {message_text}")

        # Получаем ответ
        response = client_socket.recv(buffer_size)

        if response:
            # Пытаемся декодировать ответ как cp1251
            try:
                decoded_response = response.decode('cp1251')
                return True, decoded_response
            except UnicodeDecodeError:
                # Если не получается декодировать как cp1251, возвращаем сырые байты как строку
                return True, f"[Бинарные данные: {response}]"
        else:
            return True, "[Сервер не отправил ответ]"

    except socket.timeout:
        return False, "Таймаут ожидания ответа"
    except ConnectionRefusedError:
        return False, f"Сервер {host}:{port} недоступен"
    except UnicodeEncodeError:
        return False, "Ошибка кодирования строки в cp1251"
    except socket.error as e:
        return False, f"Ошибка сети: {e}"
    except Exception as e:
        return False, f"Неизвестная ошибка: {e}"
    finally:
        client_socket.close()


In [2]:
success, response = send_cp1251_message_with_response('TST|30')
(success, response)

Отправлено: TST|30


(True, 'ERR: -2, СOM порт недоступен')

In [3]:
success, response = send_cp1251_message_with_response('DRV|30')
(success, response)

Отправлено: DRV|30


(False, 'Таймаут ожидания ответа')

In [4]:
success, response = send_cp1251_message_with_response('STS|30')
(success, response)


Отправлено: STS|30


(True, 'ERR: -2, СOM порт недоступен')

In [8]:
success, response = send_cp1251_message_with_response('CLC|02|   8400,00|14')
(success, response)


Отправлено: CLC|02|   8400,00|14


(False, 'Таймаут ожидания ответа')